# Deterministic Baseline and Local Evaluation

**Purpose.** Install the official simulator into Kaggle working storage,
load the version-controlled policy, and run legality-first self-play.

**Decision question.** Does this candidate execute reliably enough to
become a frozen control? Attach the competition data and a private dataset
containing this repository's `agent/` directory. Internet is unnecessary.

## 1. Configuration

A small smoke test catches API and packaging faults; it does not estimate
ladder strength. Increase volume only after all games terminate cleanly.

In [ ]:
from collections import Counter
from pathlib import Path
import importlib.util
import json
import shutil
import sys
import time

import pandas as pd

N_GAMES = 4
MAX_DECISIONS = 10_000
WORK_DIR = Path("/kaggle/working/agent_eval")

In [ ]:
def first_match(pattern: str) -> Path:
    matches = sorted(Path("/kaggle/input").rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"No Kaggle input matched {pattern}")
    return matches[0]

sample_dir = first_match("sample_submission/main.py").parent
candidates = [Path("../agent"), Path("agent")]
candidates += [x.parent for x in sorted(Path("/kaggle/input").rglob("main.py")) if "sample_submission" not in x.parts and "cg" not in x.parts]
repo_agent = next(
    (x for x in candidates if (x / "main.py").exists() and (x / "deck.csv").exists()),
    None,
)
print(f"Official sample: {sample_dir}")
print(f"Repository agent: {repo_agent or 'not mounted; using official sample'}")

## 2. Build an isolated environment

Copy the complete `cg` directory: its Python wrapper loads a platform
shared library. The disposable working directory is never the source of
truth for policy code.

In [ ]:
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(sample_dir, WORK_DIR)
if repo_agent:
    shutil.copy2(repo_agent / "main.py", WORK_DIR / "main.py")
    shutil.copy2(repo_agent / "deck.csv", WORK_DIR / "deck.csv")

sys.path.insert(0, str(WORK_DIR))
from cg.api import to_observation_class
from cg.game import battle_finish, battle_select, battle_start

spec = importlib.util.spec_from_file_location("candidate", WORK_DIR / "main.py")
candidate = importlib.util.module_from_spec(spec)
spec.loader.exec_module(candidate)
deck = candidate.read_deck_csv()
assert len(deck) == 60

## 3. Legality shield and game runner

Validate each action before it reaches the simulator. Failures remain in
the reportâ€”discarding them would produce a falsely optimistic estimate.

In [ ]:
def validate_action(obs_dict: dict, action: list[int]) -> None:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        assert len(action) == 60
        return
    assert isinstance(action, list)
    assert all(isinstance(index, int) for index in action)
    assert len(action) == len(set(action))
    assert obs.select.minCount <= len(action) <= obs.select.maxCount
    assert all(0 <= index < len(obs.select.option) for index in action)

def play_game() -> dict:
    started, decisions, contexts = time.perf_counter(), 0, Counter()
    try:
        obs_dict, start = battle_start(deck, deck)
        if obs_dict is None:
            return {"status": "start_error", "error_player": start.errorPlayer,
                    "error_type": start.errorType}
        while decisions < MAX_DECISIONS:
            obs = to_observation_class(obs_dict)
            if obs.current is not None and obs.current.result != -1:
                return {"status": "finished", "winner": obs.current.result,
                        "turn": obs.current.turn, "decisions": decisions,
                        "seconds": time.perf_counter() - started,
                        "contexts": dict(contexts)}
            contexts[str(getattr(obs.select, "context", "none"))] += 1
            action = candidate.agent(obs_dict)
            validate_action(obs_dict, action)
            obs_dict = battle_select(action)
            decisions += 1
        return {"status": "decision_limit", "decisions": decisions}
    except Exception as error:
        return {"status": "exception", "error": f"{type(error).__name__}: {error}",
                "decisions": decisions}
    finally:
        try:
            battle_finish()
        except Exception:
            pass

The current wrapper does not expose a random seed. Repeated self-play tests
execution, but exact seed pairing should be added if a future SDK exposes a
supported hook. For candidate comparisons, alternate policies by
`obs.current.yourIndex` and swap seats.

In [ ]:
results = []
for game in range(N_GAMES):
    result = play_game()
    result["game"] = game
    results.append(result)
    print(result)

results_df = pd.DataFrame(results)
display(results_df)
display(results_df.status.value_counts().rename("games").to_frame())
failures = results_df[results_df.status != "finished"]
print("PASS: contract smoke test" if failures.empty else "NOT READY: inspect failures")
display(failures)
Path("/kaggle/working/baseline_smoke_results.json").write_text(
    json.dumps(results, indent=2, default=str)
)

## 4. Interpretation and next experiment

Self-play balance is not a strength estimate. A frozen baseline requires
zero start errors, illegal actions, exceptions, and stalls. The first
competitive candidate should change one ideaâ€”attack ranking by knockout
and damageâ€”then face frozen opponents across both seats. Report wins,
draws, losses, failure rate, and uncertainty before using a ladder slot.